Import functions and load silver tables

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    regexp_replace,
    trim
)

marketplace = spark.table("silver_marketplace")
products = spark.table("silver_products")

print("Marketplace records:", marketplace.count())
print("Product records:", products.count())

Marketplace records: 100
Product records: 30


Checking schema of silver product and marketplace

In [0]:
marketplace.printSchema()

root
 |-- condition: string (nullable = true)
 |-- listing_date: date (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- listing_type: string (nullable = true)
 |-- matched_sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- seller: string (nullable = true)
 |-- title: string (nullable = true)



In [0]:
products.printSchema()

root
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- launch_year: integer (nullable = true)
 |-- original_price: integer (nullable = true)



In [0]:
display(
    marketplace.select(
        "listing_id",
        "title",
        "matched_sku"
    )
)


listing_id,title,matched_sku
M001,EcoBook Pro 14 Laptop 16GB,LAP1001
M002,EcoBook Pro14 i7 Used Laptop,LAP1001
M003,Eco Book Pro 14 Screen Display,LAP1001
M004,EcoBook Pro 14 Battery Original,LAP1001
M005,EcoBook Pro 14 Motherboard,LAP1001
M006,EcoBook Air 13 Laptop,LAP1002
M007,EcoBook Air13 13-inch Used,LAP1002
M008,EcoBook Air 13 Display Panel,LAP1002
M009,EcoBook Basic 15 Laptop,LAP1003
M010,EcoBook Basic15 Used Notebook,LAP1003


In [0]:
display(
    products.select(
        "sku",
        "product_name"
    )
)

sku,product_name
LAP1001,EcoBook Pro 14
LAP1002,EcoBook Air 13
LAP1003,EcoBook Basic 15
LAP1004,EcoBook Pro 16
LAP1005,EcoBook Air 15
LAP1006,EcoBook Pro 13
LAP1007,EcoBook Basic 14
LAP1008,EcoBook Pro 15
LAP1009,EcoBook Air 14
LAP1010,EcoBook Basic 13


Cleaning product name and marketplace title

In [0]:
marketplace_clean = marketplace.withColumn(
    "title_clean",
    lower(
        regexp_replace(
            trim(col("title")),
            r"[^a-zA-Z0-9]",
            " "
        )
    )
)

products_clean = products.withColumn(
    "product_name_clean",
    lower(
        regexp_replace(
            trim(col("product_name")),
            r"[^a-zA-Z0-9]",
            " "
        )
    )
)

In [0]:
display(
    marketplace_clean.select(
        "title",
        "title_clean"
    )
)

title,title_clean
EcoBook Pro 14 Laptop 16GB,ecobook pro 14 laptop 16gb
EcoBook Pro14 i7 Used Laptop,ecobook pro14 i7 used laptop
Eco Book Pro 14 Screen Display,eco book pro 14 screen display
EcoBook Pro 14 Battery Original,ecobook pro 14 battery original
EcoBook Pro 14 Motherboard,ecobook pro 14 motherboard
EcoBook Air 13 Laptop,ecobook air 13 laptop
EcoBook Air13 13-inch Used,ecobook air13 13 inch used
EcoBook Air 13 Display Panel,ecobook air 13 display panel
EcoBook Basic 15 Laptop,ecobook basic 15 laptop
EcoBook Basic15 Used Notebook,ecobook basic15 used notebook


In [0]:
display(
    products_clean.select(
        "product_name",
        "product_name_clean"
    )
)

product_name,product_name_clean
EcoBook Pro 14,ecobook pro 14
EcoBook Air 13,ecobook air 13
EcoBook Basic 15,ecobook basic 15
EcoBook Pro 16,ecobook pro 16
EcoBook Air 15,ecobook air 15
EcoBook Pro 13,ecobook pro 13
EcoBook Basic 14,ecobook basic 14
EcoBook Pro 15,ecobook pro 15
EcoBook Air 14,ecobook air 14
EcoBook Basic 13,ecobook basic 13


Create the first fuzzy candidates

In [0]:
from pyspark.sql.functions import levenshtein

candidates = marketplace_clean.alias("m").crossJoin(
    products_clean.alias("p")
).withColumn(
    "distance",
    levenshtein(
        col("m.title_clean"),
        col("p.product_name_clean")
    )
)

In [0]:
display(
    candidates.select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("p.sku").alias("candidate_sku"),
        col("p.product_name").alias("candidate_product"),
        col("distance")
    ).orderBy(
        "listing_id",
        "distance"
    )
)

listing_id,marketplace_title,candidate_sku,candidate_product,distance
M001,EcoBook Pro 14 Laptop 16GB,LAP1030,EcoBook Pro Ultra 15,10
M001,EcoBook Pro 14 Laptop 16GB,LAP1022,EcoBook Pro Lite 15,10
M001,EcoBook Pro 14 Laptop 16GB,LAP1012,EcoBook Pro Max 14,11
M001,EcoBook Pro 14 Laptop 16GB,LAP1020,EcoBook Pro X 16,11
M001,EcoBook Pro 14 Laptop 16GB,LAP1026,EcoBook Pro X 13,12
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,12
M001,EcoBook Pro 14 Laptop 16GB,LAP1016,EcoBook Pro X 14,12
M001,EcoBook Pro 14 Laptop 16GB,LAP1004,EcoBook Pro 16,12
M001,EcoBook Pro 14 Laptop 16GB,LAP1021,EcoBook Air Lite 13,13
M001,EcoBook Pro 14 Laptop 16GB,LAP1008,EcoBook Pro 15,13


Rank the candidates

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

match_window = Window.partitionBy(
    "m.listing_id"
).orderBy(
    col("distance").asc()
)

best_matches = candidates.withColumn(
    "rank",
    row_number().over(match_window)
).filter(
    col("rank") == 1
)

Showing best candidates 

In [0]:
display(
    best_matches.select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("predicted_sku"),
        col("p.product_name").alias("predicted_product"),
        col("distance")
    )
)

listing_id,marketplace_title,actual_sku,predicted_sku,predicted_product,distance
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1022,EcoBook Pro Lite 15,10
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1022,EcoBook Pro Lite 15,14
M003,Eco Book Pro 14 Screen Display,LAP1001,LAP1022,EcoBook Pro Lite 15,16
M004,EcoBook Pro 14 Battery Original,LAP1001,LAP1022,EcoBook Pro Lite 15,16
M005,EcoBook Pro 14 Motherboard,LAP1001,LAP1022,EcoBook Pro Lite 15,12
M006,EcoBook Air 13 Laptop,LAP1002,LAP1002,EcoBook Air 13,7
M007,EcoBook Air13 13-inch Used,LAP1002,LAP1021,EcoBook Air Lite 13,12
M008,EcoBook Air 13 Display Panel,LAP1002,LAP1013,EcoBook Air Plus 13,13
M009,EcoBook Basic 15 Laptop,LAP1003,LAP1003,EcoBook Basic 15,7
M010,EcoBook Basic15 Used Notebook,LAP1003,LAP1015,EcoBook Basic Plus 15,13


Create new column "match_status"

In [0]:
from pyspark.sql.functions import when, count, sum

matching_results = best_matches.withColumn(
    "match_status",
    when(
        col("m.matched_sku") == col("p.sku"),
        "Correct"
    ).otherwise("Incorrect")
)

Count of correct and incorrect matching


In [0]:
matching_results.groupBy(
    "match_status"
).count().show()

+------------+-----+
|match_status|count|
+------------+-----+
|   Incorrect|   20|
|     Correct|   80|
+------------+-----+



Matching Accuracy

In [0]:
total = matching_results.count()

correct = matching_results.filter(
    col("match_status") == "Correct"
).count()

accuracy = (correct / total) * 100

print(f"Matching Accuracy: {accuracy:.2f}%")

Matching Accuracy: 80.00%


In [0]:
product_reference = products.select(
    "sku",
    "product_name"
)

display(product_reference.orderBy("sku"))

sku,product_name
LAP1001,EcoBook Pro 14
LAP1002,EcoBook Air 13
LAP1003,EcoBook Basic 15
LAP1004,EcoBook Pro 16
LAP1005,EcoBook Air 15
LAP1006,EcoBook Pro 13
LAP1007,EcoBook Basic 14
LAP1008,EcoBook Pro 15
LAP1009,EcoBook Air 14
LAP1010,EcoBook Basic 13


Create new column model size for products and marketplace

In [0]:
from pyspark.sql.functions import regexp_extract

marketplace_features = marketplace_clean.withColumn(
    "model_size",
    regexp_extract(
        col("title_clean"),
        r"\b(13|14|15|16)\b",
        1
    )
)

products_features = products_clean.withColumn(
    "model_size",
    regexp_extract(
        col("product_name_clean"),
        r"\b(13|14|15|16)\b",
        1
    )
)

In [0]:
display(
    marketplace_features.select(
        "listing_id",
        "title",
        "model_size"
    )
)

listing_id,title,model_size
M001,EcoBook Pro 14 Laptop 16GB,14
M002,EcoBook Pro14 i7 Used Laptop,
M003,Eco Book Pro 14 Screen Display,14
M004,EcoBook Pro 14 Battery Original,14
M005,EcoBook Pro 14 Motherboard,14
M006,EcoBook Air 13 Laptop,13
M007,EcoBook Air13 13-inch Used,13
M008,EcoBook Air 13 Display Panel,13
M009,EcoBook Basic 15 Laptop,15
M010,EcoBook Basic15 Used Notebook,


In [0]:
display(
    products_features.select(
        "sku",
        "product_name",
        "model_size"
    )
)

sku,product_name,model_size
LAP1001,EcoBook Pro 14,14
LAP1002,EcoBook Air 13,13
LAP1003,EcoBook Basic 15,15
LAP1004,EcoBook Pro 16,16
LAP1005,EcoBook Air 15,15
LAP1006,EcoBook Pro 13,13
LAP1007,EcoBook Basic 14,14
LAP1008,EcoBook Pro 15,15
LAP1009,EcoBook Air 14,14
LAP1010,EcoBook Basic 13,13


Create new column size_match by comparing model_size

In [0]:
from pyspark.sql.functions import when

candidates_v2 = marketplace_features.alias("m").crossJoin(
    products_features.alias("p")
).withColumn(
    "distance",
    levenshtein(
        col("m.title_clean"),
        col("p.product_name_clean")
    )
).withColumn(
    "size_match",
    when(
        col("m.model_size") == col("p.model_size"),
        1
    ).otherwise(0)
)

In [0]:
display(
    candidates_v2.select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("p.sku").alias("candidate_sku"),
        col("p.product_name").alias("candidate_product"),
        col("distance"),
        col("size_match")
    )
)

listing_id,marketplace_title,candidate_sku,candidate_product,distance,size_match
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,12,1
M002,EcoBook Pro14 i7 Used Laptop,LAP1002,EcoBook Air 13,18,0
M003,Eco Book Pro 14 Screen Display,LAP1003,EcoBook Basic 15,20,0
M004,EcoBook Pro 14 Battery Original,LAP1004,EcoBook Pro 16,18,0
M005,EcoBook Pro 14 Motherboard,LAP1005,EcoBook Air 15,16,0
M006,EcoBook Air 13 Laptop,LAP1006,EcoBook Pro 13,10,1
M007,EcoBook Air13 13-inch Used,LAP1007,EcoBook Basic 14,15,0
M008,EcoBook Air 13 Display Panel,LAP1008,EcoBook Pro 15,18,0
M009,EcoBook Basic 15 Laptop,LAP1009,EcoBook Air 14,11,0
M010,EcoBook Basic15 Used Notebook,LAP1010,EcoBook Basic 13,15,0


Calculate token_score or similarity score


In [0]:
from pyspark.sql.functions import (
    split,
    array_intersect,
    array_union,
    size,
    when,
    col
)

candidates_v3 = (
    candidates_v2
    .withColumn(
        "marketplace_tokens",
        split(col("m.title_clean"), r"\s+")
    )
    .withColumn(
        "product_tokens",
        split(col("p.product_name_clean"), r"\s+")
    )
    .withColumn(
        "common_tokens",
        size(
            array_intersect(
                col("marketplace_tokens"),
                col("product_tokens")
            )
        )
    )
    .withColumn(
        "total_unique_tokens",
        size(
            array_union(
                col("marketplace_tokens"),
                col("product_tokens")
            )
        )
    )
    .withColumn(
        "token_score",
        when(
            col("total_unique_tokens") > 0,
            col("common_tokens") / col("total_unique_tokens")
        ).otherwise(0.0)
    )
)

In [0]:
display(
    candidates_v3
    .filter(col("m.listing_id") == "M001")
    .select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("candidate_sku"),
        col("p.product_name").alias("candidate_product"),
        col("distance"),
        col("size_match"),
        col("token_score")
    )
    .orderBy(
        col("token_score").desc()
    )
)

listing_id,marketplace_title,actual_sku,candidate_sku,candidate_product,distance,size_match,token_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1001,EcoBook Pro 14,12,1,0.6
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1012,EcoBook Pro Max 14,11,1,0.5
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1016,EcoBook Pro X 14,12,1,0.5
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1008,EcoBook Pro 15,13,0,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1006,EcoBook Pro 13,13,0,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1007,EcoBook Basic 14,15,1,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1004,EcoBook Pro 16,12,0,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1009,EcoBook Air 14,15,1,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1023,EcoBook Ultra 14,14,1,0.3333333333333333
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1018,EcoBook Studio 14,15,1,0.3333333333333333


In [0]:
display(
    candidates_v3
    .filter(col("m.listing_id") == "M002")
    .select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("candidate_sku"),
        col("p.product_name").alias("candidate_product"),
        col("distance"),
        col("size_match"),
        col("token_score")
    )
    .orderBy(
        col("token_score").desc()
    )
)

listing_id,marketplace_title,actual_sku,candidate_sku,candidate_product,distance,size_match,token_score
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1004,EcoBook Pro 16,16,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1011,EcoBook Ultra 15,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1023,EcoBook Ultra 14,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1024,EcoBook Studio 15,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1019,EcoBook Basic 16,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1001,EcoBook Pro 14,16,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1006,EcoBook Pro 13,16,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1007,EcoBook Basic 14,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1009,EcoBook Air 14,18,0,0.14285714285714285
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1018,EcoBook Studio 14,18,0,0.14285714285714285


In [0]:
from pyspark.sql.functions import greatest, lit

candidates_v3 = candidates_v3.withColumn(
    "levenshtein_similarity",
    1 - (
        col("distance") /
        greatest(
            size(col("marketplace_tokens")),
            size(col("product_tokens"))
        )
    )
)

In [0]:
candidates_v3 = candidates_v3.withColumn(
    "levenshtein_similarity",
    when(
        col("levenshtein_similarity") < 0,
        0.0
    ).otherwise(
        col("levenshtein_similarity")
    )
)

Calculate weighted score


In [0]:
candidates_v3 = candidates_v3.withColumn(
    "weighted_score",
    (
        col("token_score") * 0.50
        +
        col("levenshtein_similarity") * 0.30
        +
        col("size_match") * 0.20
    )
)

Ranking based on weighted score

In [0]:
match_window_v3 = Window.partitionBy(
    "m.listing_id"
).orderBy(
    col("weighted_score").desc()
)

best_matches_v3 = (
    candidates_v3
    .withColumn(
        "rank",
        row_number().over(match_window_v3)
    )
    .filter(
        col("rank") == 1
    )
)

In [0]:
display(
    best_matches_v3.select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("predicted_sku"),
        col("p.product_name").alias("predicted_product"),
        col("distance"),
        col("token_score"),
        col("size_match"),
        col("weighted_score")
    )
    .orderBy("listing_id")
)

listing_id,marketplace_title,actual_sku,predicted_sku,predicted_product,distance,token_score,size_match,weighted_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1001,EcoBook Pro 14,12,0.6,1,0.5
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1002,EcoBook Air 13,18,0.14285714285714285,0,0.07142857142857142
M003,Eco Book Pro 14 Screen Display,LAP1001,LAP1001,EcoBook Pro 14,16,0.2857142857142857,1,0.34285714285714286
M004,EcoBook Pro 14 Battery Original,LAP1001,LAP1001,EcoBook Pro 14,17,0.6,1,0.5
M005,EcoBook Pro 14 Motherboard,LAP1001,LAP1001,EcoBook Pro 14,12,0.75,1,0.575
M006,EcoBook Air 13 Laptop,LAP1002,LAP1002,EcoBook Air 13,7,0.75,1,0.575
M007,EcoBook Air13 13-inch Used,LAP1002,LAP1010,EcoBook Basic 13,15,0.3333333333333333,1,0.3666666666666667
M008,EcoBook Air 13 Display Panel,LAP1002,LAP1002,EcoBook Air 13,14,0.6,1,0.5
M009,EcoBook Basic 15 Laptop,LAP1003,LAP1003,EcoBook Basic 15,7,0.75,1,0.575
M010,EcoBook Basic15 Used Notebook,LAP1003,LAP1010,EcoBook Basic 13,15,0.16666666666666666,0,0.08333333333333333


Count of correct and incorrect matching

In [0]:
matching_results_v3 = best_matches_v3.withColumn(
    "match_status",
    when(
        col("m.matched_sku") == col("p.sku"),
        "Correct"
    ).otherwise("Incorrect")
)

matching_results_v3.groupBy(
    "match_status"
).count().show()

+------------+-----+
|match_status|count|
+------------+-----+
|     Correct|   90|
|   Incorrect|   10|
+------------+-----+



Matching Accuracy

In [0]:
total_v3 = matching_results_v3.count()

correct_v3 = matching_results_v3.filter(
    col("match_status") == "Correct"
).count()

accuracy_v3 = (correct_v3 / total_v3) * 100

print(f"Version 2 Matching Accuracy: {accuracy_v3:.2f}%")

Version 2 Matching Accuracy: 90.00%


 Display Incorrect Matching Results 

In [0]:
display(
    matching_results_v3
    .filter(col("match_status") == "Incorrect")
    .select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("predicted_sku"),
        col("p.product_name").alias("predicted_product"),
        col("distance"),
        col("token_score"),
        col("size_match"),
        col("weighted_score")
    )
    .orderBy("listing_id")
)

listing_id,marketplace_title,actual_sku,predicted_sku,predicted_product,distance,token_score,size_match,weighted_score
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1002,EcoBook Air 13,18,0.14285714285714285,0,0.07142857142857142
M007,EcoBook Air13 13-inch Used,LAP1002,LAP1010,EcoBook Basic 13,15,0.3333333333333333,1,0.3666666666666667
M010,EcoBook Basic15 Used Notebook,LAP1003,LAP1010,EcoBook Basic 13,15,0.16666666666666666,0,0.08333333333333333
M013,EcoBook Pro16 Used 32GB,LAP1004,LAP1014,EcoBook Studio 16,13,0.16666666666666666,0,0.08333333333333333
M016,EcoBook Air15 i5 Laptop,LAP1005,LAP1018,EcoBook Studio 14,13,0.16666666666666666,0,0.08333333333333333
M018,EcoBook Pro13 Battery,LAP1006,LAP1018,EcoBook Studio 14,13,0.2,0,0.1
M020,EcoBook Basic14 Used,LAP1007,LAP1023,EcoBook Ultra 14,11,0.2,0,0.1
M022,EcoBook Pro15 16GB Used,LAP1008,LAP1023,EcoBook Ultra 14,13,0.16666666666666666,0,0.08333333333333333
M024,EcoBook Air14 Used Notebook,LAP1009,LAP1024,EcoBook Studio 15,16,0.16666666666666666,0,0.08333333333333333
M026,EcoBook Basic13 For Parts,LAP1010,LAP1001,EcoBook Pro 14,15,0.16666666666666666,0,0.08333333333333333


 Improve Text Normalization 

In [0]:
from pyspark.sql.functions import regexp_replace

marketplace_clean_v2 = marketplace_clean.withColumn(
    "title_clean",
    regexp_replace(
        col("title_clean"),
        r"([a-zA-Z])(\d+)",
        r"$1 $2"
    )
)

products_clean_v2 = products_clean.withColumn(
    "product_name_clean",
    regexp_replace(
        col("product_name_clean"),
        r"([a-zA-Z])(\d+)",
        r"$1 $2"
    )
)

In [0]:
display(
    marketplace_clean_v2.select(
        "title",
        "title_clean"
    )
)

title,title_clean
EcoBook Pro 14 Laptop 16GB,ecobook pro 14 laptop 16gb
EcoBook Pro14 i7 Used Laptop,ecobook pro 14 i 7 used laptop
Eco Book Pro 14 Screen Display,eco book pro 14 screen display
EcoBook Pro 14 Battery Original,ecobook pro 14 battery original
EcoBook Pro 14 Motherboard,ecobook pro 14 motherboard
EcoBook Air 13 Laptop,ecobook air 13 laptop
EcoBook Air13 13-inch Used,ecobook air 13 13 inch used
EcoBook Air 13 Display Panel,ecobook air 13 display panel
EcoBook Basic 15 Laptop,ecobook basic 15 laptop
EcoBook Basic15 Used Notebook,ecobook basic 15 used notebook


Create candidate v2

In [0]:
candidates_v2_new = marketplace_clean_v2.alias("m").crossJoin(
    products_clean_v2.alias("p")
).withColumn(
    "distance",
    levenshtein(
        col("m.title_clean"),
        col("p.product_name_clean")
    )
)

Extract model size

In [0]:
from pyspark.sql.functions import regexp_extract

marketplace_features_v2 = marketplace_clean_v2.withColumn(
    "model_size",
    regexp_extract(
        col("title_clean"),
        r"\b(13|14|15|16)\b",
        1
    )
)

products_features_v2 = products_clean_v2.withColumn(
    "model_size",
    regexp_extract(
        col("product_name_clean"),
        r"\b(13|14|15|16)\b",
        1
    )
)

 Recreate Candidate V3 Using the Feature Tables 

In [0]:
candidates_v3_new = marketplace_features_v2.alias("m").crossJoin(
    products_features_v2.alias("p")
).withColumn(
    "distance",
    levenshtein(
        col("m.title_clean"),
        col("p.product_name_clean")
    )
).withColumn(
    "size_match",
    when(
        col("m.model_size") == col("p.model_size"),
        1
    ).otherwise(0)
)

Tokenize Marketplace and Product Titles ,Calculate Common and Unique Tokens,Calculate Token Score  

In [0]:
from pyspark.sql.functions import split, array_intersect, array_union, size

candidates_v3_new = (
    candidates_v3_new
    .withColumn(
        "marketplace_tokens",
        split(col("m.title_clean"), r"\s+")
    )
    .withColumn(
        "product_tokens",
        split(col("p.product_name_clean"), r"\s+")
    )
    .withColumn(
        "common_tokens",
        size(
            array_intersect(
                col("marketplace_tokens"),
                col("product_tokens")
            )
        )
    )
    .withColumn(
        "total_unique_tokens",
        size(
            array_union(
                col("marketplace_tokens"),
                col("product_tokens")
            )
        )
    )
    .withColumn(
        "token_score",
        when(
            col("total_unique_tokens") > 0,
            col("common_tokens") / col("total_unique_tokens")
        ).otherwise(0.0)
    )
)

In [0]:
display(
    candidates_v3_new
    .filter(
        col("m.listing_id").isin("M002", "M018")
    )
    .select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("candidate_sku"),
        col("p.product_name").alias("candidate_product"),
        col("distance"),
        col("size_match"),
        col("token_score")
    )
    .orderBy(
        "listing_id",
        col("token_score").desc()
    )
)

listing_id,marketplace_title,actual_sku,candidate_sku,candidate_product,distance,size_match,token_score
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1001,EcoBook Pro 14,16,1,0.42857142857142855
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1016,EcoBook Pro X 14,17,1,0.375
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1012,EcoBook Pro Max 14,17,1,0.375
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1018,EcoBook Studio 14,20,1,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1009,EcoBook Air 14,19,1,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1007,EcoBook Basic 14,20,1,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1008,EcoBook Pro 15,17,0,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1004,EcoBook Pro 16,17,0,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1023,EcoBook Ultra 14,20,1,0.25
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1006,EcoBook Pro 13,17,0,0.25


 Calculate Normalized Levenshtein Similarity 

In [0]:
from pyspark.sql.functions import greatest

candidates_v3_new = candidates_v3_new.withColumn(
    "levenshtein_similarity",
    1 - (
        col("distance") /
        greatest(
            size(col("marketplace_tokens")),
            size(col("product_tokens"))
        )
    )
)

Prevent Negative Similarity 

In [0]:
candidates_v3_new = candidates_v3_new.withColumn(
    "levenshtein_similarity",
    when(
        col("levenshtein_similarity") < 0,
        0.0
    ).otherwise(
        col("levenshtein_similarity")
    )
)

Calculated weighted score

In [0]:
candidates_v3_new = candidates_v3_new.withColumn(
    "weighted_score",
    (
        col("token_score") * 0.50
        +
        col("levenshtein_similarity") * 0.30
        +
        col("size_match") * 0.20
    )
)

Ranking candidates

In [0]:
match_window_v3_new = Window.partitionBy(
    "m.listing_id"
).orderBy(
    col("weighted_score").desc()
)

best_matches_v3_new = (
    candidates_v3_new
    .withColumn(
        "rank",
        row_number().over(match_window_v3_new)
    )
    .filter(
        col("rank") == 1
    )
)

In [0]:
display(
    best_matches_v3_new.select(
        col("m.listing_id").alias("listing_id"),
        col("m.title").alias("marketplace_title"),
        col("m.matched_sku").alias("actual_sku"),
        col("p.sku").alias("predicted_sku"),
        col("p.product_name").alias("predicted_product"),
        col("distance"),
        col("token_score"),
        col("size_match"),
        col("weighted_score")
    )
    .orderBy("listing_id")
)

listing_id,marketplace_title,actual_sku,predicted_sku,predicted_product,distance,token_score,size_match,weighted_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,LAP1001,EcoBook Pro 14,12,0.6,1,0.5
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,LAP1001,EcoBook Pro 14,16,0.42857142857142855,1,0.41428571428571426
M003,Eco Book Pro 14 Screen Display,LAP1001,LAP1001,EcoBook Pro 14,16,0.2857142857142857,1,0.34285714285714286
M004,EcoBook Pro 14 Battery Original,LAP1001,LAP1001,EcoBook Pro 14,17,0.6,1,0.5
M005,EcoBook Pro 14 Motherboard,LAP1001,LAP1001,EcoBook Pro 14,12,0.75,1,0.575
M006,EcoBook Air 13 Laptop,LAP1002,LAP1002,EcoBook Air 13,7,0.75,1,0.575
M007,EcoBook Air13 13-inch Used,LAP1002,LAP1002,EcoBook Air 13,13,0.6,1,0.5
M008,EcoBook Air 13 Display Panel,LAP1002,LAP1002,EcoBook Air 13,14,0.6,1,0.5
M009,EcoBook Basic 15 Laptop,LAP1003,LAP1003,EcoBook Basic 15,7,0.75,1,0.575
M010,EcoBook Basic15 Used Notebook,LAP1003,LAP1003,EcoBook Basic 15,14,0.6,1,0.5


 Count Correct and Incorrect Matches 

In [0]:
matching_results_v4 = best_matches_v3_new.withColumn(
    "match_status",
    when(
        col("m.matched_sku") == col("p.sku"),
        "Correct"
    ).otherwise("Incorrect")
)

matching_results_v4.groupBy(
    "match_status"
).count().show()

+------------+-----+
|match_status|count|
+------------+-----+
|     Correct|  100|
+------------+-----+



 Calculate Final Matching Accuracy 

In [0]:
total_v4 = matching_results_v4.count()

correct_v4 = matching_results_v4.filter(
    col("match_status") == "Correct"
).count()

accuracy_v4 = (correct_v4 / total_v4) * 100

print(f"Final Matching Accuracy: {accuracy_v4:.2f}%")

Final Matching Accuracy: 100.00%


Create the Final Matching Dataset 

In [0]:
final_matching = best_matches_v3_new.select(
    col("m.listing_id").alias("listing_id"),
    col("m.title").alias("marketplace_title"),
    col("p.sku").alias("matched_sku"),
    col("p.product_name").alias("matched_product"),
    col("distance"),
    col("token_score"),
    col("size_match"),
    col("weighted_score")
)

 Display the Final Matching Result 

In [0]:
display(
    final_matching.orderBy("listing_id")
)

listing_id,marketplace_title,matched_sku,matched_product,distance,token_score,size_match,weighted_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,12,0.6,1,0.5
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,EcoBook Pro 14,16,0.42857142857142855,1,0.41428571428571426
M003,Eco Book Pro 14 Screen Display,LAP1001,EcoBook Pro 14,16,0.2857142857142857,1,0.34285714285714286
M004,EcoBook Pro 14 Battery Original,LAP1001,EcoBook Pro 14,17,0.6,1,0.5
M005,EcoBook Pro 14 Motherboard,LAP1001,EcoBook Pro 14,12,0.75,1,0.575
M006,EcoBook Air 13 Laptop,LAP1002,EcoBook Air 13,7,0.75,1,0.575
M007,EcoBook Air13 13-inch Used,LAP1002,EcoBook Air 13,13,0.6,1,0.5
M008,EcoBook Air 13 Display Panel,LAP1002,EcoBook Air 13,14,0.6,1,0.5
M009,EcoBook Basic 15 Laptop,LAP1003,EcoBook Basic 15,7,0.75,1,0.575
M010,EcoBook Basic15 Used Notebook,LAP1003,EcoBook Basic 15,14,0.6,1,0.5


Save the Final Matching Result to the Silver Layer 

In [0]:
final_matching.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_fuzzy_matching")

 Verify the Saved Silver Table 

In [0]:
display(
    spark.table("workspace.default.silver_fuzzy_matching")
)

listing_id,marketplace_title,matched_sku,matched_product,distance,token_score,size_match,weighted_score
M001,EcoBook Pro 14 Laptop 16GB,LAP1001,EcoBook Pro 14,12,0.6,1,0.5
M002,EcoBook Pro14 i7 Used Laptop,LAP1001,EcoBook Pro 14,16,0.42857142857142855,1,0.41428571428571426
M003,Eco Book Pro 14 Screen Display,LAP1001,EcoBook Pro 14,16,0.2857142857142857,1,0.34285714285714286
M004,EcoBook Pro 14 Battery Original,LAP1001,EcoBook Pro 14,17,0.6,1,0.5
M005,EcoBook Pro 14 Motherboard,LAP1001,EcoBook Pro 14,12,0.75,1,0.575
M006,EcoBook Air 13 Laptop,LAP1002,EcoBook Air 13,7,0.75,1,0.575
M007,EcoBook Air13 13-inch Used,LAP1002,EcoBook Air 13,13,0.6,1,0.5
M008,EcoBook Air 13 Display Panel,LAP1002,EcoBook Air 13,14,0.6,1,0.5
M009,EcoBook Basic 15 Laptop,LAP1003,EcoBook Basic 15,7,0.75,1,0.575
M010,EcoBook Basic15 Used Notebook,LAP1003,EcoBook Basic 15,14,0.6,1,0.5
